# 02 — Hybrid Quantum GAN (multi-asset)

**ZHAW CEC Quantum Computing — Final project**

Replaces only the **generator** of the GAN with a small variational quantum circuit (PennyLane 0.44, `default.qubit` simulator). The discriminator/critic, training loop, and evaluation pipeline are reused from notebook 01 unchanged — so the only difference between this experiment and the classical baseline is the generator.

**Architecture:**
1. Latent noise (dim = `n_qubits`) encoded via `AngleEmbedding` (RY rotations)
2. `n_layers` of trainable `RY` rotations + ring-of-CNOTs entanglement
3. `n_qubits` Pauli-Z expectation values measured
4. Classical linear head + tanh maps to flat (`window * n_assets`) output

**Switches:** `MODEL_VARIANT` (`gan` / `wgan_gp`), `TICKERS` (any list), `N_QUBITS`, `N_LAYERS`, `RUN_QUBIT_SCAN` (the scaling experiment).

## Setup (Colab)

In [ ]:
import os
if not os.path.isdir('/content/qGAN-market-generator'):
    !git clone https://github.com/wuns/qGAN-market-generator.git
%cd /content/qGAN-market-generator
!pip install -q yfinance 'pennylane>=0.44,<0.45'

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd()
if (ROOT / 'src').is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'src').is_dir():
    sys.path.insert(0, str(ROOT.parent))
    ROOT = ROOT.parent

import numpy as np
import torch
import matplotlib.pyplot as plt
import pennylane as qml

from src.data            import prepare_smi_data
from src.models          import Discriminator, Critic, count_parameters
from src.quantum_models  import QuantumGenerator, count_quantum_parameters
from src.training        import set_seed, make_dataloader, build_experiment, generate
from src.evaluation      import (plot_distributions, plot_acf_comparison, plot_sample_paths,
                                 plot_correlation_comparison, build_report)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {device}')
print(f'PennyLane version: {qml.__version__}')
print(f'Repo root: {ROOT}')

## Config

In [ ]:
# Data & training (match notebook 01 for fair comparison)
TICKERS       = ['^SSMI', '^GDAXI']
MODEL_VARIANT = 'wgan_gp'             # 'gan' or 'wgan_gp'
SEED          = 42
WINDOW        = 20
EPOCHS        = 80
BATCH         = 64

# Quantum-specific
N_QUBITS = 6      # latent_dim is forced to equal this; 4-8 is reasonable on default.qubit
N_LAYERS = 3      # depth of variational circuit

# Scaling experiment (Option C): if True, train at multiple qubit counts and compare
RUN_QUBIT_SCAN = False
QUBIT_SCAN_VALUES = [4, 6, 8]
QUBIT_SCAN_EPOCHS = 60   # smaller to keep total time manageable

set_seed(SEED)

## Data

In [ ]:
tickers_in = TICKERS if isinstance(TICKERS, list) else [TICKERS]
asset_tag = '_'.join(t.replace('^', '') for t in tickers_in)

data = prepare_smi_data(
    tickers=TICKERS,
    window=WINDOW,
    cache_path=ROOT / 'results' / f'prices_{asset_tag}.pkl',
)
print(f'Tickers:       {data.tickers}')
print(f'Train windows: {data.train_windows.shape}   (n_train, window, n_assets)')
print(f'Test  windows: {data.test_windows.shape}')
print(f'Date range: {data.dates[0].date()} to {data.dates[-1].date()}  ({len(data.dates)} days)')

## Inspect the quantum circuit

Useful to render the circuit once before training, both for sanity-checking and for the presentation.

In [ ]:
from src.quantum_models import make_quantum_node
circuit, ws = make_quantum_node(N_QUBITS, N_LAYERS)
demo_input   = torch.zeros(N_QUBITS)
demo_weights = torch.zeros(*ws['weights'])
print(qml.draw(circuit)(demo_input, demo_weights))

## Build & train the hybrid quantum GAN

In [ ]:
adversary_cls = Discriminator if MODEL_VARIANT == 'gan' else Critic

exp = build_experiment(
    variant=MODEL_VARIANT,
    latent_dim=N_QUBITS,            # latent dim must match n_qubits for the quantum gen
    window=WINDOW,
    n_assets=data.n_assets,
    generator_cls=QuantumGenerator,
    adversary_cls=adversary_cls,
    generator_kwargs={'n_qubits': N_QUBITS, 'n_layers': N_LAYERS,
                      'window': WINDOW, 'n_assets': data.n_assets},
)

qparams = count_quantum_parameters(exp.generator)
n_params_A = count_parameters(exp.adversary)

print(f'Variant:                     {exp.label}')
print(f'n_qubits / n_layers:         {N_QUBITS} / {N_LAYERS}')
print(f'n_assets:                    {data.n_assets}  ({data.tickers})')
print(f'Quantum params (rotations):  {qparams["quantum_params"]}')
print(f'Classical head params:       {qparams["classical_params"]}')
print(f'Total generator params:      {qparams["total_params"]}')
print(f'{exp.adversary_role.capitalize():19s} params: {n_params_A}')

train_flat = data.flatten(data.train_windows)
loader     = make_dataloader(train_flat, batch_size=BATCH)

print('\nTraining... (quantum simulation is slower than classical; expect minutes)')
history = exp.train_fn(
    exp.generator, exp.adversary, loader,
    latent_dim=N_QUBITS, epochs=EPOCHS, device=device,
)
print(f'\nTraining time: {history.train_time_sec:.1f} s')

In [ ]:
plt.figure(figsize=(8, 3))
d_label = 'D loss' if exp.adversary_role == 'discriminator' else 'C loss'
plt.plot(history.d_loss, label=d_label)
plt.plot(history.g_loss, label='G loss')
plt.xlabel('epoch'); plt.legend()
plt.title(f'Hybrid QGAN ({exp.label}) — n_qubits={N_QUBITS}, n_layers={N_LAYERS}')
plt.tight_layout(); plt.show()

## Generate & evaluate

In [ ]:
n_eval = len(data.test_windows)
fake_flat_scaled, t_inf = generate(exp.generator, n_eval, N_QUBITS, device=device)
fake_scaled = data.unflatten(fake_flat_scaled)
fake_returns = data.unscale(fake_scaled)
real_returns = data.unscale(data.test_windows)

samples_per_sec = (n_eval * WINDOW * data.n_assets) / t_inf
print(f'Inference: {n_eval} windows in {t_inf*1000:.1f} ms ({samples_per_sec:.0f} return-samples/s)')

In [ ]:
plot_sample_paths(real_returns, fake_returns, data.tickers); plt.show()
plot_distributions(real_returns, fake_returns, data.tickers); plt.tight_layout(); plt.show()
plot_acf_comparison(real_returns, fake_returns, data.tickers); plt.tight_layout(); plt.show()
if data.n_assets >= 2:
    plot_correlation_comparison(real_returns, fake_returns, data.tickers); plt.show()

In [ ]:
report = build_report(
    real_windows=real_returns,
    fake_windows=fake_returns,
    tickers=data.tickers,
    n_params_G=qparams['total_params'],
    n_params_D=n_params_A,
    train_time_sec=history.train_time_sec,
    inference_samples_per_sec=samples_per_sec,
    extras={
        'seed':           SEED,
        'epochs':         EPOCHS,
        'window':         WINDOW,
        'n_qubits':       N_QUBITS,
        'n_layers':       N_LAYERS,
        'quantum_params': qparams['quantum_params'],
        'classical_params_in_G': qparams['classical_params'],
        'model':          f'quantum_{MODEL_VARIANT}',
    },
)

print(f"=== {report['model']}  |  assets: {report['tickers']}  |  n_qubits={N_QUBITS} ===\n")
for ticker in report['tickers']:
    r = report['real_per_asset'][ticker]; f = report['fake_per_asset'][ticker]
    print(f'{ticker}:')
    print(f'  std       real={r["std"]:.5f}  fake={f["std"]:.5f}')
    print(f'  kurtosis  real={r["kurtosis"]:.3f}    fake={f["kurtosis"]:.3f}')
    print(f'  skew      real={r["skew"]:.3f}    fake={f["skew"]:.3f}')
print(f'\noverall KS:           {report["ks_statistic_overall"]:.4f}')
if 'correlation' in report:
    print(f'corr Frobenius err:   {report["correlation"]["frobenius_err"]:.4f}')
    print(f'real corr:\n{np.array(report["correlation"]["real_corr"])}')
    print(f'fake corr:\n{np.array(report["correlation"]["fake_corr"])}')
print(f'\ntraining time:        {report["training_time_sec"]} s')
print(f'generator params:     {report["n_params_generator"]}  '
      f'({qparams["quantum_params"]} quantum + {qparams["classical_params"]} classical)')

## Save artefacts

In [ ]:
out = ROOT / 'results' / f'quantum_{MODEL_VARIANT}_{asset_tag}_q{N_QUBITS}L{N_LAYERS}'
out.mkdir(parents=True, exist_ok=True)
torch.save(exp.generator.state_dict(), out / 'generator.pt')
np.save(out / 'fake_returns.npy', fake_returns)
np.save(out / 'real_returns_test.npy', real_returns)
np.save(out / 'scale.npy', data.scale)
with open(out / 'metrics.json', 'w') as f:
    json.dump(report, f, indent=2, default=float)
print('Saved to', out)

## Optional: qubit-count scaling experiment

Set `RUN_QUBIT_SCAN = True` in the config cell to run this. Trains the same architecture at multiple qubit counts and reports how metrics scale. Each run with ~60 epochs takes a few minutes.

Note: every additional qubit doubles state-vector size (memory) and roughly doubles per-step time. Stay ≤ 10 qubits on Colab CPU.

In [ ]:
if RUN_QUBIT_SCAN:
    scan_results = []
    for nq in QUBIT_SCAN_VALUES:
        print(f'\n=== Scaling run: n_qubits = {nq} ===')
        set_seed(SEED)

        scan_exp = build_experiment(
            variant=MODEL_VARIANT,
            latent_dim=nq,
            window=WINDOW,
            n_assets=data.n_assets,
            generator_cls=QuantumGenerator,
            adversary_cls=adversary_cls,
            generator_kwargs={'n_qubits': nq, 'n_layers': N_LAYERS,
                              'window': WINDOW, 'n_assets': data.n_assets},
        )
        scan_loader = make_dataloader(train_flat, batch_size=BATCH)
        t0 = time.time()
        scan_history = scan_exp.train_fn(
            scan_exp.generator, scan_exp.adversary, scan_loader,
            latent_dim=nq, epochs=QUBIT_SCAN_EPOCHS, device=device, log_every=20,
        )
        scan_time = time.time() - t0

        fake_flat_s, _ = generate(scan_exp.generator, n_eval, nq, device=device)
        fake_s   = data.unscale(data.unflatten(fake_flat_s))
        scan_q   = count_quantum_parameters(scan_exp.generator)
        scan_rep = build_report(
            real_windows=real_returns, fake_windows=fake_s, tickers=data.tickers,
            n_params_G=scan_q['total_params'], n_params_D=count_parameters(scan_exp.adversary),
            train_time_sec=scan_time,
            inference_samples_per_sec=(n_eval * WINDOW * data.n_assets) / max(0.001, _),
            extras={'n_qubits': nq, 'n_layers': N_LAYERS,
                    'quantum_params': scan_q['quantum_params']},
        )
        scan_results.append(scan_rep)
        print(f'  n_qubits={nq}: KS={scan_rep["ks_statistic_overall"]:.4f}, '
              f"train_time={scan_time:.1f}s")

    # Save the scan summary
    scan_out = ROOT / 'results' / f'quantum_scan_{MODEL_VARIANT}_{asset_tag}'
    scan_out.mkdir(parents=True, exist_ok=True)
    with open(scan_out / 'scan_results.json', 'w') as f:
        json.dump(scan_results, f, indent=2, default=float)
    print(f'\nSaved scan to {scan_out}')
else:
    print('RUN_QUBIT_SCAN=False — skipping scaling experiment.')

In [ ]:
if RUN_QUBIT_SCAN:
    nqs   = [r['n_qubits'] for r in scan_results]
    kss   = [r['ks_statistic_overall'] for r in scan_results]
    times = [r['training_time_sec'] for r in scan_results]
    avg_kurt_real = np.mean([r['real_per_asset'][t]['kurtosis']
                              for r in scan_results for t in r['tickers']]) / len(data.tickers)
    avg_kurt_fake = [np.mean([r['fake_per_asset'][t]['kurtosis']
                              for t in r['tickers']]) for r in scan_results]

    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    ax[0].plot(nqs, kss, 'o-'); ax[0].set_xlabel('n_qubits')
    ax[0].set_ylabel('KS distance'); ax[0].set_title('Distribution match (lower better)')

    ax[1].plot(nqs, avg_kurt_fake, 'o-', label='fake')
    ax[1].axhline(np.mean([report['real_per_asset'][t]['kurtosis']
                            for t in report['tickers']]), color='C1',
                  ls='--', label='real')
    ax[1].set_xlabel('n_qubits'); ax[1].set_ylabel('avg kurtosis')
    ax[1].set_title('Tail behavior'); ax[1].legend()

    ax[2].plot(nqs, times, 'o-'); ax[2].set_xlabel('n_qubits')
    ax[2].set_ylabel('training time (s)'); ax[2].set_title('Resource cost')
    plt.tight_layout(); plt.show()

## Suggested run plan for the project

After classical baselines (notebook 01), do the following in this notebook:

1. **Smoke test:** `TICKERS='^SSMI'`, `MODEL_VARIANT='gan'`, `EPOCHS=10`, `N_QUBITS=4`, `RUN_QUBIT_SCAN=False`. 1–2 min, confirms the pipeline works.
2. **Main quantum run, univariate:** `TICKERS='^SSMI'`, `MODEL_VARIANT='wgan_gp'`, `EPOCHS=80`, `N_QUBITS=6`, `N_LAYERS=3`. Compare against `classical_wgan_gp_SSMI` from notebook 01.
3. **Main quantum run, multivariate:** `TICKERS=['^SSMI','^GDAXI']`, otherwise as above. Compare against `classical_wgan_gp_SSMI_GDAXI`.
4. **Scaling experiment:** keep multivariate setting, set `RUN_QUBIT_SCAN=True`. Produces the qubit-scaling figure.

Notebook 03 (next) will load all of these `metrics.json` files and produce the comparison tables/plots for the presentation.